In [5]:
import numpy as np
import pandas as pd
import re

In [10]:
df = (
    pd
    .read_csv("01_flats.csv")
    .drop_duplicates()    #removing duplicate rows
    .dropna(subset=['price'])
    .apply(lambda col: col.str.strip().str.lower() if col.dtype=='object' else col)
    .assign(
        bhk = lambda df_: df_['property_name'].str.extract(r"(\d{1})").astype('int8')
        ,locality = lambda df_: df_['property_name'].str.split('in ').str[1]
        ,rating = lambda df_: df_['society'].str.extract(r"(\d{1}\.{1}\d+)").astype('float')
        ,society = lambda df_: df_['society'].str.split(r"(\d{1}\.{1}\d*)").str[0]
        ,price_cr = lambda df_: df_['price'].str.replace('₹','').apply(
                                lambda x: x.replace('crore','').strip() if 'crore' in x 
                                    else round(float(x.replace('lac','').strip())/100,2) if 'lac' in x 
                                        else np.nan)
        ,bedRoom = lambda df_: df_['bedRoom'].str.extract(r"(\d+)")
        ,bathroom = lambda df_: df_['bathroom'].str.extract(r"(\d+)")
        ,balcony = lambda df_: df_['balcony'].str.extract(r"(\d*)")
        ,area_sqft = lambda df_: df_['areaWithType'].str.extract(r"(\d+)").astype('float')
        ,area_sqm = lambda df_: df_['areaWithType'].str.extract(r"(\d+\.{1}\d+)").astype('float')
        ,area = lambda df_: np.where(df_['area_sqm'].notna(), df_['area_sqm'], df_['area_sqft']*0.092903)
        ,area_type = lambda df_: df_['areaWithType'].str.split(' area').str[0].str.strip()
        ,addl_room = lambda df_: np.where(df_['additionalRoom'].notna(), 1, 0)
        ,floor_no = lambda df_: df_['floorNum'].str.extract(r"(\d+)").fillna('0').astype(int)
        ,floor = lambda df_: np.where(df_['floor_no']<4, 'low rise', np.where(df_['floor_no']<8, 'mid rise', 'high rise'))
        ,age_1 = lambda df_: df_['agePossession'].str.replace('within 3 months','0').str.
                                replace('within 6 months','0').str.findall(r"(\d+)").str[0].fillna(-1).astype(int).
                                    apply(lambda x: 2026-x if x>1000 else x)
        ,age = lambda df_: np.where(df_['age_1']<0, '-na-', np.where(df_['age_1']<2, 'new', 
                                                                     np.where(df_['age_1']<5, 'old', 'ancient')))
        ,nearby_loc = lambda df_: df_['nearbyLocations'].apply(lambda x: re.findall(r"'(.*?)'",str(x)))
    )
    .drop(columns=['property_name','link','bedRoom','property_id','price','areaWithType',
                   'area_sqft','area_sqm','additionalRoom','floorNum','floor_no','address',
                   'agePossession','age_1','description'])    #removing unnecessary columns
    .dropna(subset=['price_cr'])
    #.reindex(columns=['bhk','bathroom','balcony','addl_room','area','area_type','facing',
    #                  'floor','age','society','locality','price_cr','furnishDetails',
    #                  'features','nearbyLocations'])
)

print(df.shape)
df.sample(4)

(2996, 17)


,society,area,bathroom,balcony,facing,nearbyLocations,furnishDetails,features,rating,bhk,locality,price_cr,area_type,addl_room,floor,age,nearby_loc
1913,vatika lifestyle homes,163.04,2,3,NaN,"['sapphire 83 mall', 'nh – 08', 'knowledge tre...","['3 fan', '2 geyser', '4 light', 'no ac', 'no ...","['security / fire alarm', 'lift(s)', 'maintena...",3.8,3,sector 83 gurgaon,1.15,super built up,0,high rise,new,"[sapphire 83 mall, nh – 08, knowledge tree wor..."
1004,godrej aria,139.63,2,2,north-west,"['sapphire 83 mall', 'delhi jaipur expressway'...","['2 ac', 'no bed', 'no chimney', 'no curtains'...","['feng shui / vaastu compliant', 'security / f...",3.8,2,sector 79 gurgaon,1,super built up,1,mid rise,new,"[sapphire 83 mall, delhi jaipur expressway, mo..."
1047,aipl the peaceful homes,218.32,4,3,south,"['airia mall', 'golf corse ext. rd.', 'nationa...","['5 wardrobe', '4 fan', '1 exhaust fan', '6 ge...","['water purifier', 'centrally air conditioned'...",3.9,3,sector 70a gurgaon,2.5,super built up,1,high rise,new,"[airia mall, golf corse ext. rd., national hig..."
2775,shri ram apartments,83.61,2,1,east,"['chintapurni mandir', 'sheetla mata mandir', ...","['3 wardrobe', '4 fan', '1 exhaust fan', '10 l...","['lift(s)', 'high ceiling height', 'false ceil...",NaN,2,laxman vihar,0.46,carpet,0,low rise,new,"[chintapurni mandir, sheetla mata mandir, stat..."


In [127]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2996 entries, 0 to 3016
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   society          2995 non-null   object 
 1   area             2996 non-null   float64
 2   bathroom         2996 non-null   object 
 3   balcony          2996 non-null   object 
 4   facing           2122 non-null   object 
 5   nearbyLocations  2905 non-null   object 
 6   furnishDetails   2200 non-null   object 
 7   features         2589 non-null   object 
 8   rating           1701 non-null   float64
 9   bhk              2996 non-null   int8   
 10  locality         2996 non-null   object 
 11  price_cr         2996 non-null   object 
 12  area_type        2996 non-null   object 
 13  addl_room        2996 non-null   int64  
 14  floor            2996 non-null   object 
 15  age              2996 non-null   object 
dtypes: float64(2), int64(1), int8(1), object(12)
memory usage: 377.4+